In [1]:
rm(list = ls())

# check and install CRAN packages
cran_packages <- c("tidyverse", "ggformula", "agricolae", "ggplot2", "dplyr")

for (pkg in cran_packages) {
  if (!requireNamespace(pkg, quietly = TRUE)) {
    install.packages(pkg, dependencies = TRUE)
  }
}

lapply(cran_packages, library, character.only = TRUE)


── Attaching core tidyverse packages ─────────────────────────────────────────────────────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.6
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.1     ✔ tibble    3.3.0
✔ lubridate 1.9.4     ✔ tidyr     1.3.2
✔ purrr     1.2.0     
── Conflicts ───────────────────────────────────────────────────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
Loading required package: scales


Attaching package: ‘scales’


The following object is masked from ‘package:purrr’:

    discard


The following object is masked from ‘package:readr’:

    col_factor


Loading required package: ggiraph

Loading required package: ggridges


New to ggformula?  Try the tutorials: 
	learnr::run_tutorial("introduction", package = "ggformula")
	learn

[[1]]
 [1] "lubridate" "forcats"   "stringr"   "dplyr"     "purrr"     "readr"    
 [7] "tidyr"     "tibble"    "ggplot2"   "tidyverse" "repr"      "stats"    
[13] "graphics"  "grDevices" "utils"     "datasets"  "methods"   "base"     

[[2]]
 [1] "ggformula" "ggridges"  "ggiraph"   "scales"    "lubridate" "forcats"  
 [7] "stringr"   "dplyr"     "purrr"     "readr"     "tidyr"     "tibble"   
[13] "ggplot2"   "tidyverse" "repr"      "stats"     "graphics"  "grDevices"
[19] "utils"     "datasets"  "methods"   "base"     

[[3]]
 [1] "agricolae" "ggformula" "ggridges"  "ggiraph"   "scales"    "lubridate"
 [7] "forcats"   "stringr"   "dplyr"     "purrr"     "readr"     "tidyr"    
[13] "tibble"    "ggplot2"   "tidyverse" "repr"      "stats"     "graphics" 
[19] "grDevices" "utils"     "datasets"  "methods"   "base"     

[[4]]
 [1] "agricolae" "ggformula" "ggridges"  "ggiraph"   "scales"    "lubridate"
 [7] "forcats"   "stringr"   "dplyr"     "purrr"     "readr"     "tidyr"    
[13] "tibble"    "ggplot2"   "tidyverse" "repr"      "stats"     "graphics" 
[19] "grDevices" "utils"     "datasets"  "methods"   "base"     

[[5]]
 [1] "agricolae" "ggformula" "ggridges"  "ggiraph"   "scales"    "lubridate"
 [7] "forcats"   "stringr"   "dplyr"     "purrr"     "readr"     "tidyr"    
[13] "tibble"    "ggplot2"   "tidyverse" "repr"      "stats"     "graphics" 
[19] "grDevices" "utils"     "datasets"  "methods"   "base"

# Visualize nucleotide diversity of PPR, NLR, and TAS genes

In [2]:
# load input files
prot_temp <- read.delim2("./lib/all_gene_nuc_div.txt", row.names = 1, header = TRUE, sep = "\t")
ppr_class <- read.delim2("./lib/ppr_class.txt", sep = "\t", header = TRUE)
nlr_class <- read.delim2("./lib/nlr_class.txt", sep = "\t", header = TRUE)


In [3]:
# process PPR data
ppr_temp <- prot_temp[match(ppr_class$ID, rownames(prot_temp)), ]
ppr_temp$Class <- ppr_class$Class[match(rownames(ppr_temp), ppr_class$ID)]
ppr_temp$ID <- rownames(ppr_temp)

# process NLR data
nlr_temp <- prot_temp[match(nlr_class$ID, rownames(prot_temp)), ]
nlr_temp$Class <- nlr_class$Class[match(rownames(nlr_temp), nlr_class$ID)]
nlr_temp$Class[is.na(nlr_temp$Class)] <- "non-siRNA-NLR"
nlr_temp$ID <- rownames(nlr_temp)

prot_temp$Class <- "all-protein"
prot_temp$ID <- rownames(prot_temp)

In [4]:
# generate random data
set.seed(123)  # for reproducibility

n_sample <- 500
n_iter <- 1000

ran_pi <- ran_Tajima.D <- ran_theta <- ran_dis <- matrix(0, nrow = n_sample, ncol = n_iter)

for (i in 1:n_iter) {
  ran_ids <- sample(1:nrow(prot_temp), n_sample, replace = FALSE)
  ran_pi[, i] <- prot_temp$Pi[ran_ids]
  ran_Tajima.D[, i] <- prot_temp$Tajima.D[ran_ids]
  ran_theta[, i] <- prot_temp$theta_Watterson[ran_ids]
  ran_dis[, i] <- prot_temp$Distance[ran_ids]
}

ran_pi <- apply(ran_pi, c(1, 2), as.numeric)
ran_Tajima.D <- apply(ran_Tajima.D, c(1, 2), as.numeric)
ran_theta <- apply(ran_theta, c(1, 2), as.numeric)
ran_dis <- apply(ran_dis, c(1, 2), as.numeric)

ran_temp <- suppressWarnings(
  data.frame(
    Pi = apply(ran_pi, 1, median, na.rm = TRUE),
    Tajima.D = apply(ran_Tajima.D, 1, median, na.rm = TRUE),
    theta_Watterson = apply(ran_theta, 1, median, na.rm = TRUE),
    Distance = apply(ran_dis, 1, median, na.rm = TRUE),
    Class = "random",
    ID = paste0("random_", 1:n_sample)
  )
)

In [5]:
# combine all data
pdata <- rbind(ppr_temp, nlr_temp)
pdata <- rbind(pdata, ran_temp)

level <- c("HS-siRNA-PPR", "non-HS-siRNA-PPR", "non-siRNA-PPR", 
           "siRNA-NLR", "non-siRNA-NLR", "random")
pdata$Class <- factor(pdata$Class, levels = level)


# Calculate nucl div - theta Watterson

In [6]:
pdata$theta_Watterson <- as.numeric(pdata$theta_Watterson)

# Kruskal-Wallis test
test <- kruskal(pdata$theta_Watterson, pdata$Class, group = TRUE, p.adj = "bonferroni", alpha = 0.05)
test


$statistics
     Chisq Df p.chisq
  238.8936  5       0

$parameters
            test  p.ajusted      name.t ntr alpha
  Kruskal-Wallis bonferroni pdata$Class   6  0.05

$means
                 pdata.theta_Watterson      rank          std   r         Min
HS-siRNA-PPR                0.04399978 1050.6429 0.0187292977  14 0.015426997
non-HS-siRNA-PPR            0.03316912 1015.2500 0.0169820068   4 0.015426997
non-siRNA-NLR               0.04140664  897.8168 0.0329262485 161 0.007713499
non-siRNA-PPR               0.01298829  490.8432 0.0102754547 456 0.007713499
random                      0.01074504  517.9130 0.0002802608 500 0.010284665
siRNA-NLR                   0.02918061  782.6250 0.0215776052   4 0.010027548
                        Max        Q25        Q50        Q75
HS-siRNA-PPR     0.08017979 0.02933102 0.04047980 0.05494330
non-HS-siRNA-PPR 0.04928069 0.02032210 0.03398441 0.04683143
non-siRNA-NLR    0.14216732 0.01377411 0.02853995 0.06270328
non-siRNA-PPR    0.12968320 0.008

In [7]:
t_comp <- test$means %>% 
    rownames_to_column(var = "group") %>%
    rename(theta_Watterson = pdata.theta_Watterson) %>% 
    as_tibble() %>% 
    left_join(as_tibble(test$groups), by = c("rank" = "pdata$theta_Watterson"))

t_comp$group <- factor(t_comp$group, levels = level)
t_comp


group,theta_Watterson,rank,std,r,Min,Max,Q25,Q50,Q75,groups
<fct>,<dbl>,<dbl>,<dbl>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
HS-siRNA-PPR,0.04399978,1050.6429,0.0187292977,14,0.015426997,0.08017979,0.02933102,0.04047980,0.05494330,a
non-HS-siRNA-PPR,0.03316912,1015.2500,0.0169820068,4,0.015426997,0.04928069,0.02032210,0.03398441,0.04683143,a
non-siRNA-NLR,0.04140664,897.8168,0.0329262485,161,0.007713499,0.14216732,0.01377411,0.02853995,0.06270328,a
non-siRNA-PPR,0.01298829,490.8432,0.0102754547,456,0.007713499,0.12968320,0.00825528,0.01028467,0.01432507,b
random,0.01074504,517.9130,0.0002802608,500,0.010284665,0.01157025,0.01060606,0.01079890,0.01088766,b
siRNA-NLR,0.02918061,782.6250,0.0215776052,4,0.010027548,0.05106868,0.01096207,0.02781310,0.04603163,ab


In [8]:

p1 <- pdata %>% 
  ggplot(aes(x = Class, y = theta_Watterson, fill = Class)) +
  geom_boxplot(outlier.size = -1, width = 0.3) +
  geom_jitter(aes(group = Class), color = "black", size = 0.3,
              position = position_jitter(width = 0.2), alpha = 0.2) +
  scale_fill_manual(values = c(
    "HS-siRNA-PPR" = "#9E4231",
    "non-HS-siRNA-PPR" = "#30669B",
    "non-siRNA-PPR" = "#C6833F",
    "siRNA-NLR" = "#008073",
    "non-siRNA-NLR" = "#7C7CB1",
    "random" = "#787C7E"
  )) +
  ylim(c(0, 0.12)) +
  ylab("Nucleotide diversity, theta") +
  xlab("") +
  theme_classic() +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, vjust = 1),
    axis.ticks = element_line(colour = "black")
  )

# save plot
ggsave(p1, file = "nuc_div_theta_boxplot.pdf", width = 4.8, height = 3)


Warning message:
“Removed 10 rows containing non-finite outside the scale range (`stat_boxplot()`).”
Warning message:
“Removed 10 rows containing missing values or values outside the scale range (`geom_point()`).”


In [9]:

p2 <- gf_histogram(~ theta_Watterson | Class, alpha = 0.2, data = pdata, bins = 30) %>%
  gf_freqpoly(~ theta_Watterson, data = pdata, color = ~ Class, size = 1) +
  facet_wrap(~ Class, scales = "free_y", nrow = 1) +
  scale_colour_manual(values = c(
    "HS-siRNA-PPR" = "#9E4231",
    "non-HS-siRNA-PPR" = "#30669B",
    "non-siRNA-PPR" = "#C6833F",
    "siRNA-NLR" = "#008073",
    "non-siRNA-NLR" = "#7C7CB1",
    "random" = "#787C7E"
  )) +
  scale_x_continuous(limits = c(0, 0.15), breaks = seq(0, 0.15, 0.03)) +
  xlab("Nucleotide diversity, theta") +
  ylab("Loci Number") +
  theme_bw() +
  theme(
    panel.grid = element_blank(),
    axis.text.x = element_text(angle = 45, vjust = 1, hjust = 1),
    legend.position = "none",
    strip.background = element_rect(colour = "white", fill = "white")
  )

# save plot
ggsave(p2, file = "nuc_div_theta_density.pdf", width = 10, height = 2)



Warning message:
“Using `size` aesthetic for lines was deprecated in ggplot2 3.4.0.
ℹ Please use `linewidth` instead.”
Warning message:
“Removed 3 rows containing non-finite outside the scale range (`stat_bin()`).”
`stat_bin()` using `bins = 30`. Pick better value `binwidth`.
Warning message:
“Removed 3 rows containing non-finite outside the scale range (`stat_bin()`).”
Warning message:
“Removed 12 rows containing missing values or values outside the scale range (`geom_bar()`).”


# Calculate nucl div - distance

In [10]:
pdata$Distance <- as.numeric(pdata$Distance)

# Kruskal-Wallis test
test <- kruskal(pdata$Distance, pdata$Class, group = TRUE, p.adj = "bonferroni", alpha = 0.05)
test


$statistics
     Chisq Df p.chisq
  249.8599  5       0

$parameters
            test  p.ajusted      name.t ntr alpha
  Kruskal-Wallis bonferroni pdata$Class   6  0.05

$means
                 pdata.Distance     rank         std   r         Min
HS-siRNA-PPR        0.036076991 988.6429 0.024280952  14 0.001543210
non-HS-siRNA-PPR    0.021627816 969.7500 0.018987141   4 0.005218647
non-siRNA-NLR       0.042352614 913.1914 0.048362002 162 0.000000000
non-siRNA-PPR       0.005071959 473.1291 0.010179202 457 0.000000000
random              0.002657304 534.0740 0.000173409 500 0.002213031
siRNA-NLR           0.027940144 649.2500 0.030420318   4 0.001388171
                         Max         Q25         Q50         Q75
HS-siRNA-PPR     0.077718401 0.014669584 0.035689762 0.046358852
non-HS-siRNA-PPR 0.042511727 0.005699960 0.019390445 0.035318300
non-siRNA-NLR    0.246570122 0.005468693 0.023479726 0.063904821
non-siRNA-PPR    0.115523466 0.001036116 0.002124136 0.005640560
random         

In [11]:
t_comp <- test$means %>%
  rownames_to_column(var = "group") %>%
  rename(Distance = pdata.Distance) %>%
  as_tibble() %>%
  left_join(as_tibble(test$groups), by = c("rank" = "pdata$Distance"))

t_comp$group <- factor(t_comp$group, levels = level)
t_comp

group,Distance,rank,std,r,Min,Max,Q25,Q50,Q75,groups
<fct>,<dbl>,<dbl>,<dbl>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
HS-siRNA-PPR,0.036076991,988.6429,0.024280952,14,0.001543210,0.077718401,0.014669584,0.035689762,0.046358852,a
non-HS-siRNA-PPR,0.021627816,969.7500,0.018987141,4,0.005218647,0.042511727,0.005699960,0.019390445,0.035318300,a
non-siRNA-NLR,0.042352614,913.1914,0.048362002,162,0.000000000,0.246570122,0.005468693,0.023479726,0.063904821,a
non-siRNA-PPR,0.005071959,473.1291,0.010179202,457,0.000000000,0.115523466,0.001036116,0.002124136,0.005640560,b
random,0.002657304,534.0740,0.000173409,500,0.002213031,0.003209175,0.002526148,0.002639339,0.002767646,b
siRNA-NLR,0.027940144,649.2500,0.030420318,4,0.001388171,0.058479659,0.001975048,0.025946373,0.051911469,ab


In [12]:

p1 <- pdata %>% 
  ggplot(aes(x = Class, y = Distance, fill = Class)) +
  geom_boxplot(outlier.size = -1, width = 0.3) +
  geom_jitter(aes(group = Class), color = "black", size = 0.3,
              position = position_jitter(width = 0.2), alpha = 0.2) +
  scale_fill_manual(values = c(
    "HS-siRNA-PPR" = "#9E4231",
    "non-HS-siRNA-PPR" = "#30669B",
    "non-siRNA-PPR" = "#C6833F",
    "siRNA-NLR" = "#008073",
    "non-siRNA-NLR" = "#7C7CB1",
    "random" = "#787C7E"
  )) +
  ylim(c(0, 0.12)) +
  ylab("Nucleotide diversity, Distance") +
  xlab("") +
  theme_classic() +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, vjust = 1),
    axis.ticks = element_line(colour = "black")
  )

# save plot
ggsave(p1, file = "nuc_div_distance_boxplot.pdf", width = 4.8, height = 3)


Warning message:
“Removed 14 rows containing non-finite outside the scale range (`stat_boxplot()`).”
Warning message:
“Removed 15 rows containing missing values or values outside the scale range (`geom_point()`).”


In [13]:

p2 <- gf_histogram(~ Distance | Class, alpha = 0.2, data = pdata, bins = 30) %>%
  gf_freqpoly(~ Distance, data = pdata, color = ~ Class, size = 1) +
  facet_wrap(~ Class, scales = "free_y", nrow = 1) +
  scale_colour_manual(values = c(
    "HS-siRNA-PPR" = "#9E4231",
    "non-HS-siRNA-PPR" = "#30669B",
    "non-siRNA-PPR" = "#C6833F",
    "siRNA-NLR" = "#008073",
    "non-siRNA-NLR" = "#7C7CB1",
    "random" = "#787C7E"
  )) +
  scale_x_continuous(limits = c(0, 0.25), breaks = seq(0, 0.25, 0.05)) +
  xlab("Nucleotide diversity, Distance") +
  ylab("Loci Number") +
  theme_bw() +
  theme(
    panel.grid = element_blank(),
    axis.text.x = element_text(angle = 45, vjust = 1, hjust = 1),
    legend.position = "none",
    strip.background = element_rect(colour = "white", fill = "white")
  )

# save plot
ggsave(p2, file = "nuc_div_distance_density.pdf", width = 10, height = 2)



Warning message:
“Removed 1 row containing non-finite outside the scale range (`stat_bin()`).”
`stat_bin()` using `bins = 30`. Pick better value `binwidth`.
Warning message:
“Removed 1 row containing non-finite outside the scale range (`stat_bin()`).”
Warning message:
“Removed 12 rows containing missing values or values outside the scale range (`geom_bar()`).”
